# 02 - New Orleans Data Preparation

**Purpose:** create the New Orleans subset.

The raw Yelp review and user files are streamed line by line so the project can filter large JSONL files without loading the full dataset into memory.

## Outputs

```text
data/interim/new_orleans/businesses.jsonl
data/interim/new_orleans/reviews.jsonl
data/interim/new_orleans/users.jsonl
data/interim/new_orleans/summary.json
```

In [ ]:
# Configure paths for raw Yelp files and New Orleans interim outputs.
from pathlib import Path
import csv
import json
from collections import Counter
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_YELP_DIR = DATA_DIR / "raw" / "yelp"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

BUSINESS_PATH = RAW_YELP_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = RAW_YELP_DIR / "yelp_academic_dataset_review.json"
USER_PATH = RAW_YELP_DIR / "yelp_academic_dataset_user.json"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.name}")

## Setup

Set the selected city and output folders. From this point onward, every derived table is scoped to New Orleans.


In [ ]:
# Reuse a streaming JSONL reader so large raw files are processed safely.
TARGET_CITY = "New Orleans"
TARGET_STATE = "LA"
DATASET_SLUG = "new_orleans"

CITY_INTERIM_DIR = INTERIM_DIR / DATASET_SLUG
CITY_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

BUSINESSES_OUTPUT = CITY_INTERIM_DIR / "businesses.jsonl"
REVIEWS_OUTPUT = CITY_INTERIM_DIR / "reviews.jsonl"
USERS_OUTPUT = CITY_INTERIM_DIR / "users.jsonl"
SUMMARY_OUTPUT = CITY_INTERIM_DIR / "summary.json"

print(f"Interim output: {CITY_INTERIM_DIR.relative_to(PROJECT_ROOT)}")

In [ ]:
# Read JSONL records
def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} at line {line_number}") from exc


# Write JSONL records
def write_jsonl(records, path):
    count = 0
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False))
            file.write("\n")
            count += 1
    return count

In [ ]:
# Keep only reviews written for businesses inside the New Orleans subset.
selected_business_ids = set()

def selected_business_records():
    for record in iter_jsonl(BUSINESS_PATH):
        city = (record.get("city") or "").strip().casefold()
        state = (record.get("state") or "").strip().upper()
        if city == TARGET_CITY.casefold() and state == TARGET_STATE:
            selected_business_ids.add(record["business_id"])
            yield record

business_count = write_jsonl(selected_business_records(), BUSINESSES_OUTPUT)
print(f"Selected businesses: {business_count:,}")

In [ ]:
# Track reviewers and dates
selected_user_ids = set()
min_review_date = None
max_review_date = None


# Select reviews
def selected_review_records():
    global min_review_date, max_review_date
    for record in iter_jsonl(REVIEW_PATH):
        if record.get("business_id") in selected_business_ids:
            selected_user_ids.add(record["user_id"])
            review_date = record.get("date")
            if review_date:
                min_review_date = review_date if min_review_date is None else min(min_review_date, review_date)
                max_review_date = review_date if max_review_date is None else max(max_review_date, review_date)
            yield record


# Save reviews
review_count = write_jsonl(selected_review_records(), REVIEWS_OUTPUT)
print(f"Selected reviews: {review_count:,}")
print(f"Unique reviewing users: {len(selected_user_ids):,}")
print(f"Date range: {min_review_date} to {max_review_date}")

In [ ]:
# Select users
def selected_user_records():
    for record in iter_jsonl(USER_PATH):
        if record.get("user_id") in selected_user_ids:
            yield record


# Save users
user_count = write_jsonl(selected_user_records(), USERS_OUTPUT)
print(f"Selected user profiles: {user_count:,}")

In [ ]:
# Record an extraction summary for verification.
summary = {
    "target_city": TARGET_CITY,
    "target_state": TARGET_STATE,
    "dataset_slug": DATASET_SLUG,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "business_count": business_count,
    "review_count": review_count,
    "unique_reviewing_user_count": len(selected_user_ids),
    "matched_user_profile_count": user_count,
    "min_review_date": min_review_date,
    "max_review_date": max_review_date,
    "outputs": {
        "businesses": str(BUSINESSES_OUTPUT.relative_to(PROJECT_ROOT)),
        "reviews": str(REVIEWS_OUTPUT.relative_to(PROJECT_ROOT)),
        "users": str(USERS_OUTPUT.relative_to(PROJECT_ROOT)),
    },
}

with SUMMARY_OUTPUT.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)
    file.write("\n")

summary

## Limitations

- We will study New Orleans only, so findings should be interpreted as local community dynamics rather than general Yelp behavior.
- The extracted user table includes reviewers who interacted with New Orleans businesses, but their full Yelp history may extend beyond this city.
